# 2. Your first unbinned amplitude fit

**Learning goals:** generate a toy, fit Cartesian coefficients, read Minuit diagnostics, and compare projections and fit fractions.

Run cells from top to bottom in a fresh Python kernel. Install the package and Jupyter
as explained in [the course guide](TUTORIALS.md). No external data files are needed.
Masses are in GeV, invariants in GeV², and daughter indices start at zero.
The small event counts and grid sizes keep this lesson practical on a CPU; they are
teaching settings, not a demonstrated precision choice for a physics analysis.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

## Define the model and generate a reproducible toy

The coefficient parameters carry names, starting defaults, steps and bounds. Numeric
coefficients are fixed. Toy truth and fit starting values are kept separate so the fit
must move to describe the generated data. This model repeats lesson 1 so the notebook
runs independently.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=100, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

In [3]:
data = generate_toy(
    model, 2500, parameters=truth, seed=2026,
    method="inverse-transform", inverse_resolution=384, include_momenta=False,
)
print(f"Generated {data.size} unweighted events")
assert np.all(np.asarray(data.weights) == 1)

## Fit with the composition API

`FitSession` prepares amplitudes and normalization, constructs the unbinned likelihood,
and supplies a JAX value and gradient to iminuit. The first call includes compilation.
`simplex=True` adds a preliminary search before the gradient-based minimization.

In [4]:
session = FitSession(model, data)
start = {"NR.x": 0.35, "NR.y": 0.45}
result = session.fit(start, simplex=True, ncall=5000)
report = session.report(result)
fit_values = {name: float(result.values[name]) for name in result.parameters}
assert result.valid, "Inspect the fit diagnostics before using this result."

## Judge the result

This section uses the complete B+ -> pi+ pi- pi+ paper-style model. All numerical conventions, charge-dependent coefficients, and normalization choices follow the benchmark.

In [5]:
for label, values in [("start", start), ("truth", truth), ("fit", fit_values)]:
    print(f"NLL({label}) = {float(session.objective(values)):.6f}")
assert float(session.objective(fit_values)) <= float(session.objective(truth)) + 1e-4
for name in result.parameters:
    error = float(result.errors[name])
    print(f"{name}: truth={truth[name]:.3f}, fit={result.values[name]:.3f}, "
          f"error={error:.3f}, pull={(result.values[name]-truth[name])/error:.2f}")
print("Accurate covariance:", result.fmin.has_accurate_covar)
print("Parameters at limits:", result.fmin.has_parameters_at_limit)

## Compare data with the fitted density

This section uses the complete B+ -> pi+ pi- pi+ paper-style model. All numerical conventions, charge-dependent coefficients, and normalization choices follow the benchmark.

In [6]:
session.plot_projection(result, "s12", bins=40, projection_size=20000)
plt.show()
model.print_fit_fractions(fit_values, include_interference=True)

## Try it yourself

1. Repeat with a different starting point and compare final NLL values.
2. Increase the toy size and inspect how the errors change.
3. Plot `s13` as well; use multiple toys before interpreting a nonzero pull as bias.

## Continue learning

[Next: normalization](tutorial_03_normalization.ipynb). Reference: [fitting](../../docs/fitting.md).

Return to [the course guide](TUTORIALS.md).